# Small Llama: Held-Out Perplexity Check (One Shared Out-of-Domain Set, All 5 Fractions)

All five models (20/40/60/80/100%) are evaluated on the EXACT SAME held-out
text sample, drawn from a dataset entirely separate from Greek Wikipedia
(the training source for all five models). Since none of the five models
were trained on this dataset, this is a genuinely fair, fully comparable
generalization check across the whole learning curve, including the 100%
model, which could not be checked against unseen Wikipedia text.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import gc
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

print(f"GPU available: {torch.cuda.is_available()}")

GPU available: True


## Config

In [3]:
MODEL_DIRS = {
    "20%":  "/content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_20pct",
    "40%":  "/content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_40pct",
    "60%":  "/content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_60pct",
    "80%":  "/content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_80pct",
    "100%": "/content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_100pct",
}

OUT_OF_DOMAIN_DATASET = {
    "path": "openlanguagedata/flores_plus",
    "name": "ell_Grek",
    "split": "devtest",
    "text_field": "text",
}

N_DOCS = 200
MAX_LENGTH = 512

OUTPUT_PATH = Path("/content/drive/MyDrive/Thesis/results/small_llama_held_out_perplexity_shared_ood.json")
print("Config set")

Config set


## Load the ONE shared out-of-domain evaluation set (used for all 5 models)

In [4]:
print(f"Loading out-of-domain dataset: {OUT_OF_DOMAIN_DATASET['path']} ({OUT_OF_DOMAIN_DATASET['name']})...")

ood_ds = load_dataset(
    OUT_OF_DOMAIN_DATASET["path"],
    OUT_OF_DOMAIN_DATASET["name"],
    split=OUT_OF_DOMAIN_DATASET["split"],
    streaming=True,
)

eval_texts = []
for i, doc in enumerate(ood_ds):
    if i >= N_DOCS:
        break
    text = doc.get(OUT_OF_DOMAIN_DATASET["text_field"], "")
    if text.strip():
        eval_texts.append(text)

print(f"Shared out-of-domain evaluation set: {len(eval_texts)} documents")
print("This exact same set will be used to evaluate every model below.")

Loading out-of-domain dataset: openlanguagedata/flores_plus (ell_Grek)...


README.md:   0%|          | 0.00/73.7k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/227 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/221 [00:00<?, ?it/s]

Shared out-of-domain evaluation set: 200 documents
This exact same set will be used to evaluate every model below.


## Perplexity computation

In [5]:
def compute_perplexity(model, tokenizer, texts, max_length=MAX_LENGTH):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for text in texts:
            input_ids = tokenizer.encode(
                text, return_tensors='pt', truncation=True, max_length=max_length
            ).to(model.device)
            if input_ids.shape[1] < 2:
                continue
            outputs = model(input_ids, labels=input_ids)
            total_loss += outputs.loss.item() * input_ids.shape[1]
            total_tokens += input_ids.shape[1]
    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss)).item()
    return perplexity, avg_loss, total_tokens

## Run the check for all 5 models, on the SAME shared set

In [6]:
results = {}

if not eval_texts:
    print("No evaluation texts loaded -- fix OUT_OF_DOMAIN_DATASET above before continuing.")
else:
    for fraction_name, model_dir in MODEL_DIRS.items():
        print(f"\n{'='*70}\n{fraction_name} model\n{'='*70}")

        print(f"  Loading model from {model_dir}...")
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        model = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.float16)
        if torch.cuda.is_available():
            model = model.to('cuda')

        ppl, avg_loss, total_tokens = compute_perplexity(model, tokenizer, eval_texts)
        print(f"  Perplexity on shared out-of-domain set: {ppl:.2f} "
              f"(avg loss: {avg_loss:.4f}, {total_tokens:,} tokens)")

        results[fraction_name] = {
            'perplexity': ppl,
            'avg_loss': avg_loss,
            'tokens_evaluated': total_tokens,
        }

        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    print("\n" + "="*70)
    print("ALL MODELS DONE")
    print("="*70)


20% model
  Loading model from /content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_20pct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

  Perplexity on shared out-of-domain set: 52.92 (avg loss: 3.9688, 9,523 tokens)

40% model
  Loading model from /content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_40pct...


Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

...

  Perplexity on shared out-of-domain set: 41.39 (avg loss: 3.7230, 9,523 tokens)

80% model
  Loading model from /content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_80pct...


Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

  Perplexity on shared out-of-domain set: 39.26 (avg loss: 3.6702, 9,523 tokens)

100% model
  Loading model from /content/drive/MyDrive/Thesis/models/small_llama_batches/small_llama_100pct...


Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]

  Perplexity on shared out-of-domain set: 36.50 (avg loss: 3.5974, 9,523 tokens)

ALL MODELS DONE


## Save and summarize

In [7]:
import json

output = {
    'out_of_domain_dataset': OUT_OF_DOMAIN_DATASET,
    'n_docs': len(eval_texts),
    'results_by_fraction': results,
}

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved to {OUTPUT_PATH}\n")

print(f"{'Fraction':<10} {'Perplexity (shared out-of-domain set)'}")
for fraction_name, r in results.items():
    print(f"{fraction_name:<10} {r['perplexity']:.2f}")

Saved to /content/drive/MyDrive/Thesis/results/small_llama_held_out_perplexity_shared_ood.json

Fraction   Perplexity (shared out-of-domain set)
20%        52.92
40%        45.14
60%        41.39
80%        39.26
100%       36.50
